In [ ]:
# Make the `perovskite` package importable regardless of where Jupyter started.
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "perovskite").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

repo root: c:\Users\da1qu1r1\Desktop\AML_Perovskite_Project

# Stability Classifier

**Task:** predict synthesizability directly, as a binary label, instead of the
continuous `energy_above_hull` regression target used in `hull_regression.ipynb`.

We use the standard synthesizability cutoff:

```
is_stable = energy_above_hull <= 0.025 eV/atom
```

(see `perovskite.labels.is_stable`). The exact-zero definition used elsewhere in
this project gives only ~3.3% positive examples — a near-degenerate task where
"always predict unstable" already scores 97% plain accuracy. At 0.025 eV/atom the
positive rate rises to ~13–14%, which is still imbalanced but learnable. All
evaluation below uses **balanced accuracy** and **PR-AUC** (never plain accuracy),
and **composition-grouped CV** (`perovskite.make_group_cv`) so polymorphs of one
formula never split across train/test.

The PR-AUC chance baseline is the positive rate itself (~0.13–0.14) — a classifier
scoring at or below that is worthless.

In [ ]:
import pandas as pd
from perovskite.labels import label_sensitivity

df_meta = pd.read_csv("data/metadata_overview_reduced.csv")
print(label_sensitivity(df_meta, "stable").to_string(index=False))

 threshold  n_positive  n_negative  positive_frac  majority_baseline
     0.000         262        7738         0.0328             0.9672
     0.025        1104        6896         0.1380             0.8620
     0.050        1521        6479         0.1901             0.8099
     0.100        2160        5840         0.2700             0.7300

## Direct classifiers (grouped 3-fold CV)

Train a classifier directly on `is_stable` at the 0.025 eV/atom cutoff, for each
descriptor. `class_weight="balanced"` compensates for the ~13% positive rate.

In [ ]:
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_validate
from perovskite.data import load_features_and_meta, make_group_cv
from perovskite.labels import is_stable

THRESHOLD = 0.025
rows = []
for desc in ["soap_pca", "coulomb", "ewald"]:
    X, df = load_features_and_meta(desc)
    y = is_stable(df, THRESHOLD)
    gkf, groups = make_group_cv(df, n_splits=3)
    posrate = y.mean()

    classifiers = {
        "Dummy(stratified)": DummyClassifier(strategy="stratified", random_state=42),
        "RF(balanced)": RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                               random_state=42, n_jobs=-1),
        "HistGB(balanced)": HistGradientBoostingClassifier(class_weight="balanced",
                                                            random_state=42),
    }
    for name, clf in classifiers.items():
        r = cross_validate(clf, X, y, cv=gkf, groups=groups,
                           scoring=["balanced_accuracy", "average_precision"], n_jobs=1)
        rows.append(dict(
            descriptor=desc, model=name, positive_rate=round(posrate, 3),
            balanced_accuracy=round(r["test_balanced_accuracy"].mean(), 3),
            pr_auc=round(r["test_average_precision"].mean(), 3),
        ))

direct_results = pd.DataFrame(rows)
print(direct_results.to_string(index=False))

  descriptor             model  positive_rate  balanced_accuracy  pr_auc
    soap_pca Dummy(stratified)          0.138              0.502   0.139
    soap_pca      RF(balanced)          0.138              0.655   0.558
    soap_pca  HistGB(balanced)          0.138              0.691   0.558
     coulomb Dummy(stratified)          0.131              0.499   0.131
     coulomb      RF(balanced)          0.131              0.721   0.546
     coulomb  HistGB(balanced)          0.131              0.737   0.515
       ewald Dummy(stratified)          0.131              0.499   0.131
       ewald      RF(balanced)          0.131              0.695   0.503
       ewald  HistGB(balanced)          0.131              0.718   0.505

**Reading the table:**

- Every trained model clears the Dummy baseline (bal. acc. ~0.50, PR-AUC = chance)
  by a wide margin — there's real learnable signal for stability, not just for
  band gap.
- **Best: Coulomb + HistGB** (balanced accuracy 0.737) and **Coulomb + RF*/soap_pca +
  HistGB for PR-AUC (~0.55, roughly 4x the ~0.13 chance level)*. Unlike the
  `is_metal` task, SOAP does **not** dominate here — Coulomb and Ewald are competitive
  or better. Stability is evidently not as cleanly encoded in the (lossy, PCA'd) SOAP
  representation used here as the band-gap signal was.
- HistGB modestly beats RF for balanced accuracy across the board, consistent with
  the regression results in `hull_regression.ipynb`.
- None of these are "solved" — 0.51–0.56 PR-AUC and 0.65–0.74 balanced accuracy mean
  a real screening tool would still need a human/DFT check on flagged candidates,
  but the ranking is far better than random.

## Direct classifier: confusion matrix + learning curve

Two diagnostics for the winning family above (`HistGB(balanced)`, competitive
across all three descriptors), both respecting composition-grouped CV via
`cross_val_predict`/`learning_curve(..., groups=groups)`:

- **Confusion matrix** (out-of-fold predictions) — what precision/recall
  trade-off `class_weight="balanced"` actually buys on the `stable` class,
  as a direct comparison point against the regress-then-threshold confusion
  matrix further below.
- **Learning curve** — balanced accuracy vs. training-set size, to see whether
  the model is data-starved (CV score still climbing, train/CV gap wide) or
  has plateaued (more OQMD rows wouldn't help; better features/model would).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, classification_report

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, desc in zip(axes, ["soap_pca", "coulomb", "ewald"]):
    X, df = load_features_and_meta(desc)
    y = is_stable(df, THRESHOLD)
    gkf, groups = make_group_cv(df, n_splits=3)

    y_pred = cross_val_predict(
        HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
        X, y, cv=gkf, groups=groups, n_jobs=1,
    )
    cm = confusion_matrix(y, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, cbar=False,
                xticklabels=["pred unstable", "pred stable"],
                yticklabels=["true unstable", "true stable"])
    ax.set_title(desc)
    print(f"--- {desc} + HistGB(balanced) ---")
    print(classification_report(y, y_pred, target_names=["unstable", "stable"], digits=3))

fig.suptitle("Direct classifier confusion matrices (out-of-fold, grouped CV)")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import learning_curve

train_fracs = np.linspace(0.1, 1.0, 6)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, desc in zip(axes, ["soap_pca", "coulomb", "ewald"]):
    X, df = load_features_and_meta(desc)
    y = is_stable(df, THRESHOLD)
    gkf, groups = make_group_cv(df, n_splits=3)

    sizes, train_scores, val_scores = learning_curve(
        HistGradientBoostingClassifier(class_weight="balanced", random_state=42),
        X, y, groups=groups, cv=gkf, train_sizes=train_fracs,
        scoring="balanced_accuracy", n_jobs=-1,
    )
    train_mean, val_mean, val_std = train_scores.mean(axis=1), val_scores.mean(axis=1), val_scores.std(axis=1)
    ax.plot(sizes, train_mean, "o-", label="train")
    ax.plot(sizes, val_mean, "o-", label="CV (held-out)")
    ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15)
    ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
    ax.set_title(desc)
    ax.set_xlabel("training examples")

axes[0].set_ylabel("balanced accuracy")
axes[0].legend()
fig.suptitle("Learning curves: HistGB(balanced), grouped CV")
plt.tight_layout()
plt.show()

**What to look for when you run this:**

- **Confusion matrix:** does `class_weight="balanced"` actually catch stable
  candidates (high recall on the `stable` row), or does it still lean toward
  predicting `unstable`? The `classification_report` printed above each panel
  gives per-class precision/recall directly — compare the `stable` row's
  recall here against the regress-then-threshold confusion matrix further
  below, which only caught ~10% of true-stable structures (102/1032).
- **Learning curve:** if the CV (held-out) curve is still rising at the right
  edge and the train/CV gap hasn't closed, the model is data-starved — more
  OQMD rows (or more structures generally) would likely help. If it has
  flattened while train score stays much higher, that's overfitting/variance
  — better features or regularization would help more than raw data volume.

## Alternative: regress, then threshold

Instead of training a dedicated classifier, take the hull-energy **regressor**
(`hull_regression.ipynb`'s best model, HistGB) and threshold its continuous
prediction at 0.025 eV/atom. This uses the full continuous signal rather than a
binarized one, and only needs one model for both tasks.

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import average_precision_score, balanced_accuracy_score, confusion_matrix

THRESHOLD = 0.025
rtt_rows = []
cm_for_plot = None
for desc in ["soap_pca", "coulomb", "ewald"]:
    X, df = load_features_and_meta(desc)
    y_continuous = df["energy_above_hull"].to_numpy()
    y_true = is_stable(df, THRESHOLD)
    gkf, groups = make_group_cv(df, n_splits=3)

    y_pred_hull = cross_val_predict(HistGradientBoostingRegressor(random_state=42),
                                    X, y_continuous, cv=gkf, groups=groups, n_jobs=1)
    # lower predicted hull energy => more stable => use -y_pred_hull as the ranking score
    pr_auc = average_precision_score(y_true, -y_pred_hull)
    y_pred_stable = y_pred_hull <= THRESHOLD
    bal_acc = balanced_accuracy_score(y_true, y_pred_stable)
    rtt_rows.append(dict(descriptor=desc, balanced_accuracy=round(bal_acc, 3),
                         pr_auc=round(pr_auc, 3)))
    if desc == "coulomb":
        cm_for_plot = confusion_matrix(y_true, y_pred_stable)

rtt_results = pd.DataFrame(rtt_rows)
print(rtt_results.to_string(index=False))
print("\nconfusion matrix (Coulomb, rows=true[unstable,stable], cols=pred[unstable,stable]):")
print(cm_for_plot)

  descriptor  balanced_accuracy  pr_auc
    soap_pca              0.524   0.400
     coulomb              0.545   0.460
       ewald              0.552   0.444

confusion matrix (Coulomb, rows=true[unstable,stable], cols=pred[unstable,stable]):
[[6788   55]
 [ 930  102]]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# plot_confusion_matrix (perovskite.evaluation) expects a fitted model + X_test to
# call .predict() on; it doesn't fit cross_val_predict's precomputed out-of-fold
# predictions, so we plot this one directly.
plt.figure(figsize=(5, 4))
sns.heatmap(cm_for_plot, annot=True, fmt="d", cmap="Blues",
           xticklabels=["pred unstable", "pred stable"],
           yticklabels=["true unstable", "true stable"])
plt.title("Regress-then-threshold confusion matrix (Coulomb)")
plt.tight_layout()
plt.show()

## Conclusion: direct classification wins

| Approach | Best balanced accuracy | Best PR-AUC |
|---|---|---|
| Direct classifier (Coulomb + HistGB) | **0.737** | 0.515 |
| Direct classifier (soap_pca/Coulomb, best PR-AUC) | 0.655–0.691 | **0.558** |
| Regress-then-threshold (any descriptor) | 0.52–0.55 | 0.40–0.46 |

**Direct classification clearly beats regress-then-threshold** here — the opposite
of what we initially expected ("use the full continuous signal"). The confusion
matrix explains why: the regressor is tuned to minimize squared error on the *whole*
distribution, which is dominated by the bulk of clearly-unstable structures; it isn't
optimized for the decision boundary at 0.025 eV/atom specifically, so it misses most
of the stable class (102 of 1032 stable structures caught, recall ≈ 10%). A classifier
trained with `class_weight="balanced"` directly targets that boundary and does much
better.

**Practical recommendation:** use a dedicated classifier (Coulomb or Ewald + HistGB
for accuracy; soap_pca for PR-AUC) for a stability screen, not the hull regressor's
threshold.

**Caveats carried over from `hull_regression.ipynb`:**
- `soap_pca` is the lossy PCA-compressed SOAP (raw SOAP untested here — HistGB on
  raw SOAP is ~6 min/fold, prohibitive for a 3-descriptor x 3-model x 3-fold sweep).
- Composition-grouped CV removes polymorph leakage but not chemical-family
  extrapolation — these numbers describe ranking *new compositions within known
  chemistry*, not entirely novel chemistries.